# Laboratory Exercise 2: Hotel Bookings

Lakeside Stay is a hotel on the shore of Lake Ohrid that takes bookings directly, through online agencies, and through travel agencies. Its export in `data/bookings.csv` holds one row per booking, and `data/new-bookings.csv` counts the bookings made on each day. The bookings are synthetic and describe no real guest, hotel, or agency.

The columns of `bookings.csv` are these:

- `booking_id`, the identifier of the booking, and `booked_on`, the day it was made,
- `country`, the guest's country, and `channel`, which is direct, an online agency, or a travel agency,
- `booking_type`, an individual or a group booking, and `guests`, the number of guests as the booking system recorded it,
- `room_class`, one of economy, standard, superior, deluxe, and suite, and `room_area`, the area of the room in square metres,
- `meal_plan`, the meals included, and `payment`, which is prepaid, a deposit, or pay at hotel, with `deposit`, the amount paid in advance in euros,
- `nightly_rate`, the price of a room for one night in euros, `nights`, the length of the stay, and `lead_time_days`, the days between the booking and the arrival,
- `extras_spend`, the euros spent on extras during the stay, and
- `rating`, the guest's rating of the stay from 1 to 10, which a later model will predict.

The bookings are to be prepared for that model: every gap filled for a reason, every unusual value understood, every label turned into numbers, and every column brought to a comparable scale, with every step learned from the training bookings alone. No request asks for a model.

Answer each request in the cells below it, and add as many cells as you need. The laboratory test asks for values and decisions that your notebook computes, so keep every result visible, run the notebook from top to bottom before you submit it, and make sure it runs on a machine without internet access.

## Setup

The cells in this section are written already. Run them before the tasks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, MissingIndicator, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer, RobustScaler,
                                   StandardScaler)

DATA = Path("data")
# Every random step uses this seed, so a rerun gives the same result.
SEED = 2026

In [ ]:
# The course style.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update(
    {
        "figure.figsize": (8, 4.5),
        "figure.facecolor": GROUND,
        "figure.constrained_layout.use": True,
        "savefig.facecolor": GROUND,
        "savefig.transparent": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Archivo", "DejaVu Sans"],
        "font.size": 12,
        "text.color": INK,
        "axes.facecolor": GROUND,
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": plt.cycler(color=PALETTE),
        "grid.color": GRID,
        "xtick.color": INK,
        "ytick.color": INK,
        "legend.frameon": False,
        "image.cmap": "cividis",
    }
)
sns.set_palette(PALETTE)

In [ ]:
class FrequencyEncoder(TransformerMixin, BaseEstimator):
    """Replaces each category with its share of the rows passed to fit, and an unseen category with 0."""

    def fit(self, X: pd.DataFrame, y: object = None) -> "FrequencyEncoder":
        self.shares_ = {column: X[column].value_counts(normalize=True) for column in X.columns}
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        return np.column_stack([X[column].map(share).fillna(0).to_numpy(dtype=float)
                                for column, share in self.shares_.items()])

    def get_feature_names_out(self, input_features: object = None) -> np.ndarray:
        return np.array([f"{column}_frequency" for column in self.shares_])

In [ ]:
bookings = pd.read_csv(DATA / "bookings.csv")
new_bookings = pd.read_csv(DATA / "new-bookings.csv")
bookings.head()

## Task 1: The Gaps

### 1.1 Reading the Gaps

The file writes a value it does not have with a marker of its own. Read the bookings into a table named `df` in which every such value is missing. Count the missing values in each column, and show where they fall.

In [ ]:
pd.set_option("display.max_columns", 30)

# the file writes a missing value as "unknown"
df = pd.read_csv(DATA / "bookings.csv", na_values=["unknown"], parse_dates=["booked_on"])

gap_counts = df.isna().sum()
print("bookings with at least one gap:", df.isna().any(axis=1).sum())
gap_counts

### 1.2 Why Values Are Missing

For each column with missing values, find out whether its gaps depend on another column, whether a gap means that the value does not exist, or whether the gaps follow no pattern. Show the comparison that decides each case.

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(df.isna().astype(int), cmap=["#ffffff", PALETTE[1]], cbar=False, yticklabels=False)
plt.grid(False)
plt.xticks(rotation=45, ha="right")
plt.title("Where the values are missing")
plt.show()# Write your solution here.

### 1.3 What Deleting Would Cost

Find out how many bookings would remain if every booking with a missing value were dropped, and how the mix of channels would change. Then decide what to do with the bookings that state no rating, and apply the decision to `df`.

In [ ]:
# Write your solution here.

## Task 2: The Split and the Fills

### 2.1 The Test Bookings

The bookings made on 1 July 2026 or later stand in for bookings that the preparation has never seen. Set them aside as the test bookings, `X_test` and `y_test`, and keep the earlier bookings as the training bookings, `X_train` and `y_train`, where `X` holds every column except the rating and `y` holds the rating. Every later step learns only from the training bookings. Report the number of each.

In [ ]:
# Write your solution here.

### 2.2 Filling the Gaps

Fill the gaps of each column in the way its pattern calls for, learning every fill from the training bookings, and keep a record of the filled values wherever the absence itself carries information. Report the value that each fill by a single value uses.

In [ ]:
# Write your solution here.

### 2.3 How Well a Fill Recovers the Rate

Hide the known nightly rate of every training booking whose identifier is divisible by five. Fill these rates back in three ways, all from the numeric columns: with the median rate, from the five most similar bookings judged on standardized columns, and by predicting each column from the others in rounds with the seed 2026. Measure how far each fill lies from the true rates on average, and show the filled rates against the true ones.

In [ ]:
# Write your solution here.

### 2.4 Gaps in Time

`new_bookings` counts the bookings made on each day, and a day without a row was not recorded. Make every day of the period visible, then fill each gap from the neighbouring days, at most three days into a gap and only between recorded days. Report how many days are missing before the fill and how many stay missing after it, and show the series around its longest gap.

In [ ]:
# Write your solution here.

## Task 3: Outliers

### 3.1 Unusual Numbers of Guests

Flag the unusual numbers of guests among the training bookings in two ways: as more than 1.5 interquartile ranges beyond the quartiles, and as more than three standard deviations from the mean. Report how many bookings each way flags, show them, and explain why the two ways disagree.

In [ ]:
# Write your solution here.

### 3.2 Errors or Genuine Values

Decide for each flagged number of guests whether it is an error or a genuine value, and handle it accordingly in `X`, `X_train`, and `X_test`.

In [ ]:
# Write your solution here.

### 3.3 Unusual Combinations

Among the training bookings of individual guests with a known nightly rate, find the bookings whose combination of room area and nightly rate is unusual, in two ways: by how quickly random splits isolate a booking, and by how much sparser its neighbourhood is than the neighbourhoods of its neighbours. Judge both on the two columns standardized, treat 2% of the bookings as unusual, use 20 neighbours, and use the seed 2026. Decide what to do with the bookings that both ways flag.

In [ ]:
# Write your solution here.

### 3.4 Capping the Nights

Find the 1st and the 99th percentiles of the number of nights among the training bookings, and count the test bookings that lie beyond them. Decide whether to cap the nights at these bounds.

In [ ]:
# Write your solution here.

## Task 4: Encoding and Scaling

### 4.1 Labels as Numbers

Turn every label into numbers in the way its kind calls for, learning from the training bookings:

- the room class keeps its order, from economy to suite,
- a label without an order gets one column per category, a single column when it has two categories, and zeros for a category that the training bookings never held, and
- the country becomes its share of the training bookings, with 0 for a country that the training bookings never held.

Report the code of each room class, the number of columns that the labels without an order become, and the share of Serbia.

In [ ]:
# Write your solution here.

### 4.2 Comparable Scales

Bring every numeric column to a scale that suits it, and reduce the skew of the columns with a long right tail. Report the skew of the lead time before and after, the power that brings it closest to a normal distribution, and what the extras spend needs instead.

In [ ]:
# Write your solution here.

## Task 5: One Pipeline and Leakage

### 5.1 One Preparation

Join every step of Tasks 2 to 4 into one preparation that is fitted on the training bookings and only applied to the test bookings. Report the number of rows and columns of the prepared training and test tables, and confirm that they hold no missing values.

In [ ]:
# Write your solution here.

### 5.2 Fitted on All Bookings

Fit the same preparation on all bookings, the test bookings included, and compare what it learns for the countries that only the test bookings hold with what the preparation fitted on the training bookings learns. Then show how the leak changes the test bookings on the number of nights. Standardize the known test nights once with the mean and the standard deviation of the training bookings and once with those of all bookings, compare how far they spread, and count the test bookings beyond the 99th percentile of the nights of the training bookings and of all bookings.

In [ ]:
# Write your solution here.

### 5.3 The Decisions

Record every preparation decision in a table, with the reason the data give for it.

### 5.3 The Decisions

Record every preparation decision in a table, with the reason the data give for it.